# [Google Colab · GPU] Multilingual sentiment: Models A and B (approved run)

**Approved scope (2026-10-01):**
- **Data:** TeSent (Telugu) + TweetEval sentiment (English) + CMTET (Telugu-English code-mixed). The files are byte-identical to the audited versions (SHA-256 checked).
- **Label mappings:** only the documented ones. TeSent labels are already Negative/Neutral/Positive. TweetEval uses `mapping.txt` (0 = negative, 1 = neutral, 2 = positive). CMTET uses NEG→Negative, NTL→Neutral, POS→Positive. The original label is kept in `source_label`.
- **Cleaning and split:** remove exact duplicate text-label pairs, merge, then make a 70/15/15 split stratified by label × language_type.
- **Models:** A = TF-IDF + Logistic Regression, B = `xlm-roberta-base` without context.
- **Model C:** Skipped — no real conversation_id/message_order available.
- **Outputs:** a new folder, `MyDrive/NLP/outputs_multilingual_AB/run_<timestamp>/`. The Telugu-only outputs on the local PC are never touched.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

## 0. Setup and configuration

In [ ]:
# Colab: install libraries and mount Google Drive (datasets and outputs live on Drive so they survive runtime resets)
!pip -q install evaluate datasets accelerate tabulate   # installs only what is missing; no upgrades, so no runtime restart
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi

In [ ]:
import os, re, json, html, glob, shutil, random, inspect, unicodedata, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             precision_recall_fscore_support, classification_report,
                             confusion_matrix)

import torch
import transformers
import datasets
import evaluate
from transformers import (XLMRobertaTokenizerFast, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback,
                          DataCollatorWithPadding, set_seed)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*symlinks.*")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)
seed_everything()

USE_CUDA = torch.cuda.is_available()
print("pandas", pd.__version__, "| numpy", np.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| datasets", datasets.__version__,
      "| evaluate", evaluate.__version__)
print("CUDA available:", USE_CUDA, "| fp16 will be", "ENABLED" if USE_CUDA else "DISABLED (CPU run)")

In [ ]:
# ===== CONFIGURATION (Colab GPU). APPROVED SCOPE: TeSent + TweetEval + CMTET, Models A and B only =====
import hashlib, zipfile
from datetime import datetime
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a GPU runtime: Runtime -> Change runtime type -> T4 GPU.")

DRIVE_NLP = Path("/content/drive/MyDrive/NLP")
INPUT_ZIP = DRIVE_NLP / "multilingual_inputs_verified.zip"   # built on the local PC from the audited files
INPUT_DIR = Path("/content/multilingual_inputs")
PREPARED_DIR = Path("/content/prepared")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

# SHA-256 of every audited input file (D:\NLP\multilingual_inputs, audit of 2026-10-01).
EXPECTED_SHA256 = {
    "telugu_tesent/README.md": "3e05bcf571abf0cf0ffa0c769d66f89c746bea8c14cdfd26641e7580e960471f",
    "telugu_tesent/Tesent.csv": "ca532a16d7150ce18b5139ce055998390ec7a748cc1f1bb941dba69c44a1a50a",
    "english_tweeteval/mapping.txt": "c0572796fbe261d8ea5fd16c5061076fbba49bc69fc865d6abccaf60daacee24",
    "english_tweeteval/README.md": "cf91d41af5748a4a9eb6914327cdcc6585113c8c1caf16bc39a1b93500450b6b",
    "english_tweeteval/test_labels.txt": "6afb4afe9374d1f983bcf9a7c79b108d0f37fdf020a83f30488309bed215db9d",
    "english_tweeteval/test_text.txt": "09a93a55c63fd93f97485ef7302889d7edb4091cd49733aa37da094f0bfa0675",
    "english_tweeteval/train_labels.txt": "122bfb1732fb6995b0e5c5f726c0ba457c469c3b6e60513007ce5037f23e65d4",
    "english_tweeteval/train_text.txt": "368f01052ea6fd8ffc408a2a2e6ac9669e31542581a0396ef16591ea26eb98a6",
    "english_tweeteval/val_labels.txt": "b4566926c72e2e4e2916c864def94e76c4cdde52446af2c7ba4fc2006e057e51",
    "english_tweeteval/val_text.txt": "e5b021e6fc45064c260b09814b803d8f56cada519c4d952d72f43d48a350a964",
    "code_mixed_cmtet/codemix_sentiment_data.txt": "ac1b59e1146530c39d42e0ea37f0d714344bf49c570c5d7ba45787f5805a144c",
    "code_mixed_cmtet/README.md": "ddc136003a741c549959f8a0c439b03e77b04de555cb2d9833c32a39c9aaa6fe"
}

TESENT_DIR, TWEETEVAL_DIR, CMTET_DIR = INPUT_DIR / "telugu_tesent", INPUT_DIR / "english_tweeteval", INPUT_DIR / "code_mixed_cmtet"
TELUGU_CSV_PATH     = TESENT_DIR / "Tesent.csv"
ENGLISH_CSV_PATH    = PREPARED_DIR / "tweeteval_sentiment_mapped.csv"   # written below from the official files
CMTET_TXT_PATH      = CMTET_DIR / "codemix_sentiment_data.txt"
CMTET_README_PATH   = CMTET_DIR / "README.md"
CODE_MIXED_CSV_PATH = PREPARED_DIR / "cmtet_parsed_after_audit.csv"     # written only if the CMTET parser audit passes

DATASETS = OrderedDict({
    "telugu":     {"path": TELUGU_CSV_PATH,     "language_type": "telugu"},
    "english":    {"path": ENGLISH_CSV_PATH,    "language_type": "english"},
    "code_mixed": {"path": CODE_MIXED_CSV_PATH, "language_type": "code_mixed"},
})
SECONDARY_FILES = {}
USER_VERIFIED_MAPPINGS = {}
MAX_ROWS_PER_DATASET = None        # approved: use all verified data
ALLOW_MODEL_C = False              # approved scope: Models A and B only

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_DIR = DRIVE_NLP / "outputs_multilingual_AB" / f"run_{RUN_ID}"   # new folder per run: nothing is overwritten
CHECKPOINT_DIR = Path("/content/checkpoints")
CLEANUP_CHECKPOINTS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
print("Outputs ->", OUTPUT_DIR)

# ---- get the input ZIP (asks for it once if it is not on Drive yet), unpack, verify checksums
if not INPUT_ZIP.exists():
    from google.colab import files
    print(r"Select D:\NLP\colab_upload\multilingual_inputs_verified.zip from your PC")
    up = files.upload()
    INPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)
    INPUT_ZIP.write_bytes(up[next(iter(up))])
    print("Saved to Drive:", INPUT_ZIP)
with zipfile.ZipFile(INPUT_ZIP) as z:
    z.extractall(INPUT_DIR)
bad = {}
for rel, want in EXPECTED_SHA256.items():
    f = INPUT_DIR / rel
    got = hashlib.sha256(f.read_bytes()).hexdigest() if f.exists() else "MISSING"
    if got != want:
        bad[rel] = got
if bad:
    raise SystemExit(f"Input files differ from the audited versions -> stopping. {bad}")
print(f"All {len(EXPECTED_SHA256)} input files match the audited SHA-256 checksums.")
docs = OUTPUT_DIR / "source_docs"
for d in [TESENT_DIR, TWEETEVAL_DIR, CMTET_DIR]:
    (docs / d.name).mkdir(parents=True, exist_ok=True)
    for fn in ["SOURCE.json", "README.md", "mapping.txt"]:
        if (d / fn).exists(): shutil.copy(d / fn, docs / d.name / fn)

In [ ]:
# English: TweetEval sentiment from the OFFICIAL files. The mapping is read from mapping.txt; nothing is hard-coded.
def _lines(p):
    txt = Path(p).read_text(encoding="utf-8")
    out = [l.rstrip("\r") for l in txt.split("\n")]
    return out[:-1] if txt.endswith("\n") else out

tw_map = {}
for line in (TWEETEVAL_DIR / "mapping.txt").read_text(encoding="utf-8").splitlines():
    m = re.match(r"^\s*(\d+)\s+(\w+)\s*$", line)
    if m: tw_map[m.group(1)] = m.group(2).lower()
assert sorted(tw_map.values()) == ["negative", "neutral", "positive"], f"unexpected mapping.txt: {tw_map}"
print("TweetEval mapping.txt:", tw_map)
parts = []
for split in ["train", "val", "test"]:
    t, l = _lines(TWEETEVAL_DIR / f"{split}_text.txt"), _lines(TWEETEVAL_DIR / f"{split}_labels.txt")
    assert len(t) == len(l), f"{split}: {len(t)} texts vs {len(l)} labels"
    lab = [x.strip() for x in l]
    assert set(lab) <= set(tw_map), f"{split}: labels not in mapping.txt"
    parts.append(pd.DataFrame({"text": t, "label": [tw_map[x].capitalize() for x in lab],
                               "source_label": lab, "original_split": split}))
tw_df = pd.concat(parts, ignore_index=True)
tw_df.to_csv(ENGLISH_CSV_PATH, index=False, encoding="utf-8")
print(f"TweetEval: {len(tw_df)} rows written to {ENGLISH_CSV_PATH} (source_label keeps the original 0/1/2)")
print(pd.crosstab(tw_df["source_label"], tw_df["label"]))

In [ ]:
# Code-mixed: CMTET. The original codemix_sentiment_data.txt is parsed directly.
# Format (official README): "NEG|NTL|POS: <sentence>" followed by a line of per-word language tags (EN/TE/NE/UNIV).
# Mapping accepted: NEG->Negative, NTL->Neutral, POS->Positive only. The original tag is kept in source_label.
from collections import Counter

# CMTET official format (README of github.com/ksubbu199/cmtet-sentiment):
#   "<NEG|NTL|POS>: <sentence>"  followed by a line of per-word language tags (EN/TE/NE/UNIV), blocks separated by blank lines.
CMTET_TAG_MAP = {"NEG": "Negative", "NTL": "Neutral", "POS": "Positive"}   # the only mapping accepted
CMTET_LANG_TAGS = {"en", "te", "ne", "univ"}
CMTET_PAPER_COUNTS = {"POS": 7925, "NEG": 7713, "NTL": 4219}             # RANLP 2021 paper, Sec. 3.3 (total 19,857)
SENT_RE = re.compile(r"^(NEG|NTL|POS):\s?(.*)$")
ANY_TAG_RE = re.compile(r"^([A-Z]{2,5}):\s")


def parse_cmtet(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    records, issues = [], Counter()
    unparsed_examples, unknown_tags = [], Counter()
    i, n = 0, len(lines)
    while i < n:
        line = lines[i].rstrip("\r")
        if not line.strip():
            i += 1
            continue
        m = SENT_RE.match(line)
        if not m:
            t = ANY_TAG_RE.match(line)
            if t:
                unknown_tags[t.group(1)] += 1
            issues["unparsed_line"] += 1
            if len(unparsed_examples) < 10:
                unparsed_examples.append({"line_no": i + 1, "text": line[:200]})
            i += 1
            continue
        tag, sentence = m.group(1), m.group(2).strip()
        # next non-blank line should be the language-tag line
        j = i + 1
        while j < n and not lines[j].strip():
            j += 1
        lang_tags, n_lang = None, None
        if j < n and not SENT_RE.match(lines[j]) and set(lines[j].lower().split()) <= CMTET_LANG_TAGS:
            lang_tags = lines[j].strip().lower()
            n_lang = len(lang_tags.split())
            nxt = j + 1
        else:
            issues["missing_language_tag_line"] += 1
            nxt = i + 1
        if not sentence:
            issues["empty_sentence"] += 1
        if n_lang is not None and n_lang != len(sentence.split()):
            issues["token_vs_language_tag_count_mismatch"] += 1
        records.append({"line_no": i + 1, "source_label": tag, "label_name": CMTET_TAG_MAP[tag],
                        "text": sentence, "language_tags": lang_tags})
        i = nxt
    df = pd.DataFrame(records)
    counts = df["source_label"].value_counts().to_dict() if len(df) else {}
    audit = {
        "source_file": str(path), "total_lines": n, "parsed_sentences": len(df),
        "source_label_counts": counts,
        "paper_reported_counts": CMTET_PAPER_COUNTS,
        "difference_vs_paper": {k: counts.get(k, 0) - v for k, v in CMTET_PAPER_COUNTS.items()},
        "tag_mapping_used": CMTET_TAG_MAP,
        "unknown_sentence_tags": dict(unknown_tags),
        "issues": dict(issues), "unparsed_examples": unparsed_examples,
        "duplicate_text_label_pairs": int(df.duplicated(["text", "source_label"]).sum()) if len(df) else 0,
        "same_text_conflicting_tags": int((df.groupby("text")["source_label"].nunique() > 1).sum()) if len(df) else 0,
    }
    return df, audit


def cmtet_audit_passes(audit, max_unparsed_ratio=0.005):
    problems = []
    if audit["parsed_sentences"] == 0:
        problems.append("no sentences parsed")
    if audit["unknown_sentence_tags"]:
        problems.append(f"unknown sentence tags present: {audit['unknown_sentence_tags']}")
    if set(audit["source_label_counts"]) - set(CMTET_TAG_MAP):
        problems.append("labels outside NEG/NTL/POS")
    unparsed = audit["issues"].get("unparsed_line", 0)
    if audit["parsed_sentences"] and unparsed / audit["parsed_sentences"] > max_unparsed_ratio:
        problems.append(f"{unparsed} unparsed lines (> {max_unparsed_ratio:.1%} of sentences)")
    return problems


CMTET_CITATION = """Kusampudi Siva Subrahamanyam Varma, Preetham Sathineni, Radhika Mamidi. 2021.
Sentiment Analysis in Code-Mixed Telugu-English Text with Unsupervised Data Normalization.
In Proceedings of RANLP 2021, pages 753-760. https://doi.org/10.26615/978-954-452-072-4_086
Data: https://github.com/ksubbu199/cmtet-sentiment
"""

CMTET_READY = False
if not CMTET_TXT_PATH.exists():
    print(f"CMTET source not found: {CMTET_TXT_PATH}")
else:
    cmtet_df, cmtet_audit = parse_cmtet(CMTET_TXT_PATH)
    cmtet_problems = cmtet_audit_passes(cmtet_audit)
    cmtet_audit["audit_passed"] = not cmtet_problems
    cmtet_audit["audit_problems"] = cmtet_problems
    cmtet_audit["readme_found"] = CMTET_README_PATH.exists()
    print(json.dumps({k: v for k, v in cmtet_audit.items() if k != "unparsed_examples"}, indent=1))

    src_out = OUTPUT_DIR / "cmtet_source"
    src_out.mkdir(parents=True, exist_ok=True)
    if CMTET_README_PATH.exists():
        shutil.copy(CMTET_README_PATH, src_out / "README.md")
    (src_out / "CITATION.txt").write_text(CMTET_CITATION, encoding="utf-8")
    with open(OUTPUT_DIR / "cmtet_parser_audit.json", "w", encoding="utf-8") as f:
        json.dump(cmtet_audit, f, ensure_ascii=False, indent=2)

    if cmtet_problems:
        print("CMTET AUDIT FAILED -> no CSV exported:", cmtet_problems)
    elif not CMTET_README_PATH.exists():
        print("CMTET README.md missing next to the .txt -> no CSV exported (the README documents the NEG/NTL/POS tags).")
    else:
        out = cmtet_df.rename(columns={"label_name": "label"})[["text", "label", "source_label", "language_tags", "line_no"]]
        out.to_csv(CODE_MIXED_CSV_PATH, index=False, encoding="utf-8")
        CMTET_READY = True
        print(f"Audit passed -> exported {len(out)} rows to {CODE_MIXED_CSV_PATH}")
        display(out.head())

In [ ]:
# ---------------- READINESS GATE (approved scope) ----------------
problems = []
if not TELUGU_CSV_PATH.exists(): problems.append("TeSent missing")
if not ENGLISH_CSV_PATH.exists(): problems.append("TweetEval CSV was not prepared")
if not CMTET_READY: problems.append("CMTET parser audit did not pass")
if problems:
    raise SystemExit(f"Readiness gate failed: NO training will run. {problems}")
print("Ready: TeSent + TweetEval + CMTET verified. Training scope: Model A and Model B only.")
print("Model C (context-aware): Skipped — no real conversation_id/message_order available.")

In [ ]:
LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = [ID2LABEL[i] for i in range(3)]
LANGUAGES = ["telugu", "english", "code_mixed"]

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128

for k, v in DATASETS.items():
    print(f"{k:11s} -> {v['path']}  (exists: {Path(v['path']).exists()})")

## 1. Data audit

For every file this section prints: path, shape, columns and dtypes, 5 sample rows, missing values, duplicates, the detected text and label columns, original label counts, whether conversation metadata exists, and whether the labels are already valid Negative/Neutral/Positive labels.

**Numeric labels** are mapped only if the dataset's own README/documentation explicitly states the mapping. Otherwise that dataset is **excluded from training**. The same happens to **non-sentiment label schemes** (e.g. hate / non-hate), which are never forced into three sentiment classes.

In [ ]:
TEXT_COL_CANDIDATES  = ["text", "content", "comment", "comments", "sentence", "message", "tweet", "review", "utterance"]
LABEL_COL_CANDIDATES = ["label", "labels", "sentiment", "class", "target", "polarity"]
CONV_ID_CANDIDATES   = ["conversation_id", "conv_id", "dialogue_id", "dialog_id", "thread_id", "chat_id"]
MSG_ORDER_CANDIDATES = ["message_order", "turn", "turn_id", "message_index", "utterance_index", "msg_order"]
VALID_SENTIMENT = {"negative": "Negative", "neutral": "Neutral", "positive": "Positive"}


def read_table(path):
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        xl = pd.ExcelFile(path)
        sheets = xl.sheet_names
        return xl.parse(sheets[0]), {"sheets": sheets, "sheet_used": sheets[0]}
    return pd.read_csv(path, encoding="utf-8"), {}


def _match_col(cols, candidates):
    low = {c.lower().strip(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    return None


def detect_text_column(df):
    col = _match_col(df.columns, TEXT_COL_CANDIDATES)
    if col is not None:
        return col, "name match"
    obj = [c for c in df.columns if df[c].dtype == object or str(df[c].dtype).startswith("str")]
    if not obj:
        return None, "no string columns"
    lens = {c: df[c].astype(str).str.len().mean() for c in obj}
    return max(lens, key=lens.get), "longest average string column (heuristic)"


def detect_label_column(df, text_col):
    col = _match_col(df.columns, LABEL_COL_CANDIDATES)
    if col is not None:
        return col, "name match"
    return None, "no column named like a label"


def find_documentation(path):
    # Only documentation files that sit in the dataset's own folder are considered.
    folder = Path(path).parent
    docs = []
    for pat in ["README*", "readme*", "*.md", "*.txt", "*.rst", "DATASET*", "*card*"]:
        docs.extend(folder.glob(pat))
    return sorted(set(docs))


def find_documented_numeric_mapping(doc_texts, numeric_labels):
    # Look for explicit statements such as '0 = Negative', '0: negative', 'label 0 -> Negative'.
    mapping = {}
    for text in doc_texts:
        for num in numeric_labels:
            pat = rf"(?<![\w.]){re.escape(str(num))}\s*(?:=|:|->|=>|–|-|means|is)\s*[\"'`*]*(negative|neutral|positive)\b"
            m = re.search(pat, text, flags=re.IGNORECASE)
            if m and num not in mapping:
                mapping[num] = VALID_SENTIMENT[m.group(1).lower()]
    ok = set(mapping) == set(numeric_labels) and len(set(mapping.values())) == len(mapping)
    return (mapping if ok else None), mapping


def assess_labels(series):
    vals = series.dropna()
    uniq = sorted(vals.unique().tolist(), key=str)
    numeric = pd.api.types.is_numeric_dtype(vals) or all(re.fullmatch(r"-?\d+(\.0+)?", str(u).strip()) for u in uniq)
    if numeric:
        return {"kind": "numeric", "uniques": [int(float(u)) for u in uniq], "valid_sentiment": False}
    norm = {str(u).strip().lower() for u in uniq}
    valid = norm.issubset(VALID_SENTIMENT) and len(norm) >= 2
    return {"kind": "string", "uniques": [str(u) for u in uniq], "valid_sentiment": bool(valid),
            "all_three_present": norm == set(VALID_SENTIMENT)}


def to_py(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)): return o.tolist()
    if isinstance(o, Path): return str(o)
    return str(o)

In [ ]:
def audit_file(name, path, language_type, primary=True):
    path = Path(path)
    print("=" * 100)
    print(f"DATASET: {name}  |  primary training file: {primary}")
    print(f"File name: {path.name}\nPath: {path}")
    rep = {"dataset": name, "file_name": path.name, "path": str(path), "language_type": language_type,
           "primary_file": primary}
    if not path.exists():
        print("FILE NOT FOUND")
        rep.update(eligible_for_training=False, decision="file not found")
        return rep, None

    df, meta = read_table(path)
    rep.update(meta)
    if meta.get("sheets") and len(meta["sheets"]) > 1:
        print(f"Workbook sheets: {meta['sheets']} -> using '{meta['sheet_used']}'")
    print(f"Shape: {df.shape}")
    print("\nColumns and dtypes:"); print(df.dtypes.to_string())
    print("\nFirst 5 rows:"); display(df.head(5))
    miss = df.isna().sum()
    print("Missing values per column:"); print(miss.to_string())
    dup_rows = int(df.duplicated().sum())
    print(f"Duplicate rows (all columns): {dup_rows}")

    text_col, text_how = detect_text_column(df)
    label_col, label_how = detect_label_column(df, text_col)
    print(f"\nDetected text column : {text_col!r} ({text_how})")
    print(f"Detected label column: {label_col!r} ({label_how})")
    conv_col = _match_col(df.columns, CONV_ID_CANDIDATES)
    order_col = _match_col(df.columns, MSG_ORDER_CANDIDATES)
    print(f"conversation_id column exists: {conv_col is not None} ({conv_col})")
    print(f"message_order column exists  : {order_col is not None} ({order_col})")

    rep.update(shape=list(df.shape), columns={c: str(t) for c, t in df.dtypes.items()},
               missing_values=miss.to_dict(), duplicate_rows=dup_rows,
               text_column=text_col, text_column_detection=text_how,
               label_column=label_col, label_column_detection=label_how,
               conversation_id_column=conv_col, message_order_column=order_col,
               has_conversation_metadata=bool(conv_col and order_col))
    if text_col is not None:
        rep["duplicate_text_label_pairs"] = int(df.duplicated([text_col] + ([label_col] if label_col else [])).sum())

    if label_col is None:
        print("\nNo label column -> this file cannot be used for supervised training.")
        rep.update(eligible_for_training=False, label_mapping=None,
                   decision="No label column found; unlabeled file.")
        return rep, df

    counts = df[label_col].value_counts(dropna=False)
    print("\nOriginal label values and counts:"); print(counts.to_string())
    rep["original_label_counts"] = {str(k): int(v) for k, v in counts.items()}
    la = assess_labels(df[label_col])
    rep["label_kind"] = la["kind"]
    rep["labels_already_valid_sentiment"] = la["valid_sentiment"]
    print(f"Labels already valid Negative/Neutral/Positive: {la['valid_sentiment']}")

    docs = find_documentation(path)
    rep["documentation_files_found"] = [str(d) for d in docs]
    print(f"Documentation files in dataset folder: {[d.name for d in docs] or 'NONE'}")

    mapping, decision = None, ""
    if la["kind"] == "string" and la["valid_sentiment"]:
        mapping = {u: VALID_SENTIMENT[str(u).strip().lower()] for u in la["uniques"]}
        decision = "Labels are explicit sentiment strings; mapped by name to the target scheme."
        if not la.get("all_three_present"):
            decision += " WARNING: not all three classes are present."
    elif la["kind"] == "numeric":
        print(f"Numeric labels found: {la['uniques']}")
        doc_texts = [d.read_text(encoding="utf-8", errors="ignore") for d in docs]
        verified, partial = find_documented_numeric_mapping(doc_texts, la["uniques"])
        if verified:
            mapping = verified
            decision = f"Numeric mapping explicitly verified in documentation: {[d.name for d in docs]}"
        elif name in USER_VERIFIED_MAPPINGS and USER_VERIFIED_MAPPINGS[name].get("source"):
            um = USER_VERIFIED_MAPPINGS[name]
            mapping = {int(k): v for k, v in um["mapping"].items()}
            decision = f"Numeric mapping supplied by user with documentation source: {um['source']}"
        else:
            decision = ("TRAINING STOPPED for this dataset: labels are numeric "
                        f"{la['uniques']} and no documentation in the dataset folder explicitly maps them to "
                        "Negative/Neutral/Positive"
                        + (f" (documentation files searched: {[d.name for d in docs]}; partial matches: {partial})"
                           if docs else " (no README/documentation file was supplied)")
                        + ". Guessing a mapping (e.g. 0=Negative) would risk silently training on wrong labels.")
    else:
        decision = ("TRAINING STOPPED for this dataset: labels "
                    f"{la['uniques']} are not a Negative/Neutral/Positive sentiment scheme. "
                    "They cannot be converted to three sentiment classes without inventing semantics "
                    "(e.g. 'hate' is not equivalent to 'Negative sentiment', and 'non-hate' can be positive, "
                    "neutral, or negative).")

    rep["label_mapping"] = {str(k): v for k, v in mapping.items()} if mapping else None
    rep["eligible_for_training"] = bool(mapping) and primary
    rep["decision"] = decision if primary else decision + " (secondary file: audited only, not used for training)"
    print(f"\nDECISION: {rep['decision']}")
    return rep, df


audit_report = {"target_classes": ID2LABEL, "datasets": [], "secondary_files": []}
raw_frames = {}
for name, cfg in DATASETS.items():
    rep, df = audit_file(name, cfg["path"], cfg["language_type"], primary=True)
    audit_report["datasets"].append(rep)
    raw_frames[name] = (df, rep)
for name, paths in SECONDARY_FILES.items():
    for p in paths:
        rep, _ = audit_file(name, p, DATASETS[name]["language_type"], primary=False)
        audit_report["secondary_files"].append(rep)

In [ ]:
# Extra check for Telugu: the TeSent README describes official split files (Train/Val/Test) and a full file.
tel_dir = Path(TELUGU_CSV_PATH).parent
split_files = {p.stem: p for p in tel_dir.glob("*.csv") if p.stem.lower() in {"train", "val", "test"}}
if split_files:
    full = pd.read_csv(TELUGU_CSV_PATH)
    union = set()
    for s, p in split_files.items():
        union |= set(pd.read_csv(p)["Content"])
    note = (f"Telugu folder also contains official splits {sorted(split_files)}; "
            f"{len(union & set(full['Content']))} of {full['Content'].nunique()} unique sentences in "
            f"{Path(TELUGU_CSV_PATH).name} appear in them. The full file is used and re-split 70/15/15 as the project requires. "
            f"README states 22,505 sentences; the supplied full file has {len(full)} rows.")
    print(note)
    audit_report["datasets"][0]["notes"] = note

summary = pd.DataFrame([{
    "dataset": r["dataset"], "file": r["file_name"], "rows": (r.get("shape") or [None])[0],
    "text_col": r.get("text_column"), "label_col": r.get("label_column"),
    "labels": r.get("original_label_counts"), "valid_sentiment_labels": r.get("labels_already_valid_sentiment"),
    "conversation_metadata": r.get("has_conversation_metadata"), "eligible_for_training": r.get("eligible_for_training"),
} for r in audit_report["datasets"] + audit_report["secondary_files"]])
display(summary)

eligible = [r["dataset"] for r in audit_report["datasets"] if r["eligible_for_training"]]
excluded = {r["dataset"]: r["decision"] for r in audit_report["datasets"] if not r["eligible_for_training"]}
audit_report["eligible_datasets"] = eligible
audit_report["excluded_datasets"] = excluded
print("\nEligible for training:", eligible)
for k, v in excluded.items():
    print(f"EXCLUDED {k}: {v}")

with open(OUTPUT_DIR / "audit_report.json", "w", encoding="utf-8") as f:
    json.dump(audit_report, f, ensure_ascii=False, indent=2, default=to_py)
print(f"\nSaved {OUTPUT_DIR / 'audit_report.json'}")

if not eligible:
    raise SystemExit("No dataset has verified three-class sentiment labels. Training aborted; see audit_report.json.")

## 2. Data cleaning

Standard schema: `raw_text, cleaned_text, label, language_type, conversation_id, message_order, context_text, source_dataset`
(plus an auxiliary `annotations` column. TeSent supplies this and it is used only for error analysis).

Cleaning **removes only** URLs, HTML tags/entities, control characters and repeated whitespace.
It **keeps** Telugu Unicode (zero-width joiners included), Romanized Telugu and English words, emojis, `!`/`?`, and every word, including the negations *not, no, never, ledu, kaadu, avvale, vaddu*.
There is no lowercasing, stop-word removal or stemming.

In [ ]:
URL_RE  = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
HTML_TAG_RE = re.compile(r"</?[A-Za-z][^<>]{0,200}>")
WS_RE = re.compile(r"\s+")
NEGATIONS = ["not", "no", "never", "ledu", "kaadu", "avvale", "vaddu"]


def clean_text(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None
    t = str(text)
    t = html.unescape(t)                       # &amp; &quot; &#39; ... -> characters
    t = HTML_TAG_RE.sub(" ", t)                # <br>, <p>, </a> ...
    t = URL_RE.sub(" ", t)
    # Drop control characters (Unicode category Cc) and the BOM. Keep ZWJ/ZWNJ (category Cf):
    # Telugu conjuncts and emoji sequences depend on them.
    t = "".join(" " if unicodedata.category(ch) == "Cc" else ch for ch in t).replace("\ufeff", "")
    t = WS_RE.sub(" ", t).strip()
    return t if t else None


# Self-checks for the cleaning guarantees
_tests = ["I am not happy!! <br> see https://x.com/a", "Idi ledu kaadu avvale vaddu 😡 ?",
          "నాకు ఇష్టం లేదు 😢", "never   no\tway &amp; 👍🏽"]
for s in _tests:
    print(repr(s), "->", repr(clean_text(s)))
assert all(w in clean_text("not no never ledu kaadu avvale vaddu").split() for w in NEGATIONS)
assert "😢" in clean_text("నాకు ఇష్టం లేదు 😢") and "లేదు" in clean_text("నాకు ఇష్టం లేదు 😢")

In [ ]:
def build_standard(name, df, rep):
    text_col, label_col = rep["text_column"], rep["label_column"]
    raw_mapping = rep["label_mapping"]
    mapping = {}
    for k, v in raw_mapping.items():
        mapping[k] = v
        if re.fullmatch(r"-?\d+", k):
            mapping[int(k)] = v
    conv_col, order_col = rep.get("conversation_id_column"), rep.get("message_order_column")
    lang = DATASETS[name]["language_type"]
    stats = OrderedDict(initial_rows=len(df))

    out = pd.DataFrame({
        "raw_text": df[text_col],
        "orig_label": df[label_col],
        "conversation_id": df[conv_col] if conv_col else np.nan,
        "message_order": df[order_col] if order_col else np.nan,
    })
    out["annotations"] = df["Annotations"] if "Annotations" in df.columns else np.nan
    out["source_label"] = df["source_label"] if "source_label" in df.columns else df[label_col].astype(str)
    out["language_tags"] = df["language_tags"] if "language_tags" in df.columns else np.nan
    out["language_type"] = df["language_type"].astype(str).str.strip().str.lower() if lang == "per_row" else lang

    n = len(out); out = out[out["raw_text"].notna() & (out["raw_text"].astype(str).str.strip() != "")]
    stats["removed_missing_text"] = n - len(out)
    n = len(out); out = out[out["orig_label"].notna()]
    stats["removed_missing_label"] = n - len(out)

    def map_label(v):
        key = v.strip() if isinstance(v, str) else v
        if isinstance(key, float) and key.is_integer(): key = int(key)
        if key in mapping: return LABEL2ID[mapping[key]]
        return LABEL2ID.get(VALID_SENTIMENT.get(str(key).strip().lower(), ""), np.nan)
    out["label"] = out["orig_label"].map(map_label)
    n = len(out); out = out[out["label"].notna()]
    stats["removed_unmappable_label"] = n - len(out)
    out["label"] = out["label"].astype(int)

    out["cleaned_text"] = out["raw_text"].map(clean_text)
    n = len(out); out = out[out["cleaned_text"].notna()]
    stats["removed_empty_after_cleaning"] = n - len(out)

    n = len(out); out = out.drop_duplicates(subset=["cleaned_text", "label"], keep="first")
    stats["removed_exact_duplicate_text_label_pairs"] = n - len(out)

    if MAX_ROWS_PER_DATASET and len(out) > MAX_ROWS_PER_DATASET:
        n = len(out)
        out, _ = train_test_split(out, train_size=MAX_ROWS_PER_DATASET, stratify=out["label"], random_state=SEED)
        stats["removed_by_row_cap"] = n - len(out)

    stats["final_rows"] = len(out)
    conflicts = out.groupby("cleaned_text")["label"].nunique()
    stats["texts_with_conflicting_labels_kept"] = int((conflicts > 1).sum())

    out["source_dataset"] = f"{name}:{Path(rep['path']).name}"
    out["context_text"] = None
    cols = ["raw_text", "cleaned_text", "label", "language_type", "conversation_id",
            "message_order", "context_text", "source_dataset", "annotations", "source_label", "language_tags"]
    return out[cols].reset_index(drop=True), stats


frames, cleaning_stats = [], {}
for name in eligible:
    df, rep = raw_frames[name]
    std, stats = build_standard(name, df, rep)
    cleaning_stats[name] = stats
    frames.append(std)
    print(f"\n[{name}] cleaning stages:")
    for k, v in stats.items():
        print(f"  {k:45s} {v}")

data = pd.concat(frames, ignore_index=True)
print(f"\nCombined cleaned rows: {len(data)}")
print(pd.crosstab(data["language_type"], data["label"].map(ID2LABEL), margins=True))
neg_hits = {w: int(data["cleaned_text"].str.contains(rf"\b{w}\b", case=False, regex=True).sum()) for w in NEGATIONS}
print("\nNegation-word occurrences preserved in cleaned_text:", neg_hits)

## 3. Context creation

Format: `[PREVIOUS_2] <msg t-2> [PREVIOUS_1] <msg t-1> [CURRENT] <msg t>`, using only **earlier** messages of the **same** conversation, after sorting by `message_order`. Future messages are never used.

If `conversation_id` / `message_order` are unavailable, `context_text = cleaned_text`.

In [ ]:
def build_context(df):
    df = df.copy()
    has_meta = df["conversation_id"].notna() & df["message_order"].notna()
    df["context_text"] = df["cleaned_text"]
    if has_meta.any():
        sub = df[has_meta].sort_values(["conversation_id", "message_order"], kind="mergesort")
        g = sub.groupby("conversation_id", sort=False)["cleaned_text"]
        p1, p2 = g.shift(1), g.shift(2)          # shift(k>0) looks only backward -> no future leakage
        ctx = []
        for cur, a, b in zip(sub["cleaned_text"], p1, p2):
            parts = []
            if isinstance(b, str): parts.append(f"[PREVIOUS_2] {b}")
            if isinstance(a, str): parts.append(f"[PREVIOUS_1] {a}")
            parts.append(f"[CURRENT] {cur}")
            ctx.append(" ".join(parts))
        df.loc[sub.index, "context_text"] = ctx
    return df, int(has_meta.sum())


# Synthetic demonstration of the context logic. NOT training data; it only verifies the format.
demo = pd.DataFrame({"conversation_id": ["c1"] * 4 + ["c2"] * 2, "message_order": [3, 1, 4, 2, 1, 2],
                     "cleaned_text": ["m3", "m1", "m4", "m2", "x1", "x2"]})
demo_ctx, _ = build_context(demo)
display(demo_ctx.sort_values(["conversation_id", "message_order"]))
assert demo_ctx.loc[1, "context_text"] == "[CURRENT] m1"
assert demo_ctx.loc[3, "context_text"] == "[PREVIOUS_1] m1 [CURRENT] m2"
assert demo_ctx.loc[2, "context_text"] == "[PREVIOUS_2] m2 [PREVIOUS_1] m3 [CURRENT] m4"
assert demo_ctx.loc[4, "context_text"] == "[CURRENT] x1"

data, n_meta_rows = build_context(data)
n_ctx_rows = int((data["context_text"] != data["cleaned_text"]).sum())
# Model C is only a distinct experiment when at least some messages actually receive earlier-message context.
TRUE_CONTEXT = n_ctx_rows > 0
RUN_CONTEXT_MODEL = TRUE_CONTEXT and globals().get("ALLOW_MODEL_C", True)
if n_meta_rows == 0:
    print("True context-aware training cannot be performed because conversation_id/message_order is unavailable.")
print(f"rows with conversation metadata: {n_meta_rows} of {len(data)}; "
      f"rows where context_text differs from cleaned_text: {n_ctx_rows}")
print("Model C (context-aware XLM-R) will be", "TRAINED" if RUN_CONTEXT_MODEL else
      "SKIPPED: context_text == cleaned_text for every row, so it would duplicate Model B")
display(data.head(5))

data.to_csv(OUTPUT_DIR / "cleaned_all_data.csv", index=False, encoding="utf-8-sig")
label_mapping_out = {
    "target_classes": ID2LABEL,
    "per_dataset_original_to_target": {r["dataset"]: r["label_mapping"] for r in audit_report["datasets"]},
    "verification": {r["dataset"]: r["decision"] for r in audit_report["datasets"]},
}
with open(OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping_out, f, ensure_ascii=False, indent=2, default=to_py)
print("Saved cleaned_all_data.csv and label_mapping.json")

## 4. Train / validation / test split (70 / 15 / 15)

Stratification is joint on `label × language_type`. When conversation IDs exist, `StratifiedGroupKFold` with 20 folds is used and folds are allocated 14/3/3, so **no conversation spans two splits**.

In [ ]:
def split_data(df):
    df = df.copy()
    df["strata"] = df["label"].astype(str) + "_" + df["language_type"]
    has_groups = df["conversation_id"].notna().any()
    if has_groups:
        # Rows without a conversation (e.g. single-sentence corpora) become their own one-row group.
        groups = np.where(df["conversation_id"].notna(),
                          df["source_dataset"] + "::" + df["conversation_id"].astype(str),
                          "single::" + df.index.astype(str))
        sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
        fold = np.zeros(len(df), dtype=int)
        for i, (_, idx) in enumerate(sgkf.split(df, df["strata"], groups=groups)):
            fold[idx] = i
        split = np.where(fold < 14, "train", np.where(fold < 17, "validation", "test"))
        method = "StratifiedGroupKFold(20) -> folds 0-13 train, 14-16 validation, 17-19 test"
    else:
        strat = df["strata"]
        if strat.value_counts().min() < 7:   # need enough members per stratum for both splits
            strat = df["label"]
            print("Joint label x language stratification infeasible (tiny strata); falling back to label-only.")
        tr_idx, tmp_idx = train_test_split(df.index, test_size=0.30, stratify=strat, random_state=SEED)
        va_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, stratify=strat.loc[tmp_idx], random_state=SEED)
        split = pd.Series("train", index=df.index)
        split.loc[va_idx] = "validation"; split.loc[te_idx] = "test"
        split = split.values
        method = "train_test_split stratified on label x language_type (no conversation IDs -> not group-aware)"
    df["split"] = split
    return df.drop(columns="strata"), method, has_groups


data_split, split_method, group_aware = split_data(data)
train_df = data_split[data_split.split == "train"].drop(columns="split").reset_index(drop=True)
val_df   = data_split[data_split.split == "validation"].drop(columns="split").reset_index(drop=True)
test_df  = data_split[data_split.split == "test"].drop(columns="split").reset_index(drop=True)

print("Split method:", split_method)
for n, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{n:10s} {len(d):6d} rows ({len(d)/len(data_split):.1%})  label dist: "
          f"{d['label'].map(ID2LABEL).value_counts(normalize=True).round(3).to_dict()}")
print(pd.crosstab(data_split["split"], [data_split["language_type"], data_split["label"].map(ID2LABEL)]))

# ---- Leakage checks
print("\nLEAKAGE CHECKS")
if group_aware:
    s = {n: set((d["source_dataset"] + "::" + d["conversation_id"].astype(str))[d["conversation_id"].notna()]) for n, d in [("train", train_df), ("validation", val_df), ("test", test_df)]}
    for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        print(f"  conversation_id overlap {a} & {b}: {len(s[a] & s[b])}")
    assert not (s["train"] & s["validation"] or s["train"] & s["test"] or s["validation"] & s["test"])
else:
    print("  conversation_id unavailable -> conversation leakage check N/A (no conversations exist to leak).")
tx = {n: set(d["cleaned_text"]) for n, d in [("train", train_df), ("validation", val_df), ("test", test_df)]}
text_overlap = {f"{a}&{b}": len(tx[a] & tx[b]) for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]}
print("  identical cleaned_text across splits (only possible for same text with different labels):", text_overlap)

for n, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    d.to_csv(OUTPUT_DIR / f"{n}.csv", index=False, encoding="utf-8-sig")
audit_report["pipeline"] = {"cleaning_stats": cleaning_stats, "true_context_available": TRUE_CONTEXT,
                            "rows_with_context": n_ctx_rows, "context_model_trained": RUN_CONTEXT_MODEL,
                            "split_method": split_method, "split_sizes": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
                            "cross_split_identical_text": text_overlap}
print("Saved train.csv, validation.csv, test.csv")

## 5–6. Shared evaluation utilities

Every model is evaluated the same way on the untouched test split: overall and per-class metrics, the classification report, the confusion matrix (PNG + CSV), per-language metrics, prediction confidence (max class probability), and misclassified examples with **heuristic** error reasons.

In [ ]:
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\U0001F1E6-\U0001F1FF]")
TELUGU_RE = re.compile("[\u0C00-\u0C7F]")
LATIN_RE = re.compile("[A-Za-z]")
NEG_RE = re.compile(r"\b(" + "|".join(NEGATIONS) + r")\b|లేదు|కాదు|వద్దు|కాలేదు|లేను", re.IGNORECASE)


def heuristic_error_reason(row, has_context):
    reasons = []
    text = row["cleaned_text"]
    if row["confidence"] < 0.5: reasons.append("low confidence prediction")
    if NEG_RE.search(text): reasons.append("negation handling")
    if EMOJI_RE.search(text): reasons.append("emoji ambiguity")
    if row["language_type"] == "code_mixed" or (TELUGU_RE.search(text) and LATIN_RE.search(text)):
        reasons.append("mixed-language ambiguity")
    row_has_ctx = has_context and row.get("context_text") != text
    if len(text.split()) <= 3 and not row_has_ctx: reasons.append("insufficient conversational context")
    ann = row.get("annotations")
    if isinstance(ann, str) and len({a.split("-")[0].strip() for a in ann.split("|")}) > 1:
        reasons.append("annotation ambiguity")
    if {row["actual_id"], row["pred_id"]} == {0, 2} and not reasons:
        reasons.append("sarcasm/implicit sentiment")
    if 1 in (row["actual_id"], row["pred_id"]) and not reasons:
        reasons.append("class overlap")
    return "; ".join(reasons) if reasons else "class overlap"


def per_class_table(y, p):
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=[0, 1, 2], zero_division=0)
    return {ID2LABEL[i]: {"precision": float(pr[i]), "recall": float(rc[i]), "f1": float(f1[i]), "support": int(sup[i])} for i in range(3)}


def evaluate_model(tag, display_name, input_col, test_df, probs, best_val_macro_f1):
    y = test_df["label"].values
    pred = probs.argmax(1); conf = probs.max(1)
    m = {
        "model": display_name, "input": input_col, "n_test": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=[0, 1, 2], zero_division=0)),
        "weighted_f1": float(f1_score(y, pred, average="weighted", labels=[0, 1, 2], zero_division=0)),
        "macro_precision": float(precision_score(y, pred, average="macro", labels=[0, 1, 2], zero_division=0)),
        "macro_recall": float(recall_score(y, pred, average="macro", labels=[0, 1, 2], zero_division=0)),
        "per_class": per_class_table(y, pred),
        "classification_report": classification_report(y, pred, labels=[0, 1, 2], target_names=CLASS_NAMES, zero_division=0, output_dict=True),
        "confidence": {"mean": float(conf.mean()), "mean_when_correct": float(conf[pred == y].mean()) if (pred == y).any() else None,
                       "mean_when_wrong": float(conf[pred != y].mean()) if (pred != y).any() else None},
        "best_validation_macro_f1": best_val_macro_f1,
    }
    print(f"==== {display_name} — TEST RESULTS (input: {input_col}) ====")
    print(classification_report(y, pred, labels=[0, 1, 2], target_names=CLASS_NAMES, digits=4, zero_division=0))
    print(f"accuracy={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}  weighted_f1={m['weighted_f1']:.4f}  "
          f"mean confidence={m['confidence']['mean']:.4f}")

    # Per-language
    per_lang = {}
    for lang in LANGUAGES:
        mask = (test_df["language_type"] == lang).values
        if mask.sum() == 0:
            per_lang[lang] = {"status": "not available: no verified sentiment data for this language", "n": 0,
                              "accuracy": None, "macro_f1": None, "weighted_f1": None}
            continue
        yl, pl = y[mask], pred[mask]
        per_lang[lang] = {"status": "evaluated", "n": int(mask.sum()), "accuracy": float(accuracy_score(yl, pl)),
                          "macro_f1": float(f1_score(yl, pl, average="macro", labels=[0, 1, 2], zero_division=0)),
                          "weighted_f1": float(f1_score(yl, pl, average="weighted", labels=[0, 1, 2], zero_division=0)),
                          "per_class": per_class_table(yl, pl)}
    conv_mask = test_df["conversation_id"].notna().values
    if conv_mask.any():
        m["conversation_subset"] = {"n": int(conv_mask.sum()),
                                    "accuracy": float(accuracy_score(y[conv_mask], pred[conv_mask])),
                                    "macro_f1": float(f1_score(y[conv_mask], pred[conv_mask], average="macro", labels=[0, 1, 2], zero_division=0))}
        print("Conversation-message subset:", m["conversation_subset"])
    m["per_language"] = per_lang
    print("Per-language:", {k: (round(v["macro_f1"], 4) if v["macro_f1"] is not None else v["status"]) for k, v in per_lang.items()})

    # Confusion matrix
    cm = confusion_matrix(y, pred, labels=[0, 1, 2])
    pd.DataFrame(cm, index=[f"true_{c}" for c in CLASS_NAMES], columns=[f"pred_{c}" for c in CLASS_NAMES]) \
      .to_csv(OUTPUT_DIR / f"confusion_matrix_{tag}.csv", encoding="utf-8-sig")
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"{display_name}\nTest confusion matrix")
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / f"confusion_matrix_{tag}.png", dpi=150)
    plt.show(); plt.close(fig)

    # Predictions + misclassified
    has_ctx = input_col == "context_text"
    pred_df = test_df.copy()
    pred_df["actual_id"], pred_df["pred_id"], pred_df["confidence"] = y, pred, conf
    for i, c in enumerate(CLASS_NAMES):
        pred_df[f"prob_{c}"] = probs[:, i]
    pred_df["actual_label"] = pred_df["actual_id"].map(ID2LABEL)
    pred_df["predicted_label"] = pred_df["pred_id"].map(ID2LABEL)
    pred_df.to_csv(OUTPUT_DIR / f"predictions_{tag}.csv", index=False, encoding="utf-8-sig")

    wrong = pred_df[pred_df["actual_id"] != pred_df["pred_id"]].copy()
    wrong["likely_error_reason"] = wrong.apply(heuristic_error_reason, axis=1, has_context=has_ctx)
    wrong["reason_type"] = "HEURISTIC (rule-based guess, not ground truth)"
    # up to 15 examples: the most confident mistakes interleaved with low-confidence ones, one per reason where possible
    wrong = wrong.sort_values("confidence", ascending=False)
    k = min(15, len(wrong))
    picked = pd.concat([wrong.head((k + 1) // 2), wrong.tail(k // 2)]).drop_duplicates()
    if len(picked) < k:
        picked = pd.concat([picked, wrong.drop(picked.index).head(k - len(picked))])
    cols = ["actual_label", "predicted_label", "confidence", "raw_text", "cleaned_text"] + \
           (["context_text"] if input_col == "context_text" else []) + \
           ["language_type", "likely_error_reason", "reason_type"]
    mis = picked[cols].reset_index(drop=True)
    mis.to_csv(OUTPUT_DIR / f"misclassified_{tag}.csv", index=False, encoding="utf-8-sig")
    m["n_misclassified_total"] = int(len(wrong))
    m["heuristic_error_reason_counts_all_mistakes"] = wrong["likely_error_reason"].str.split("; ").explode().value_counts().to_dict()
    print(f"Misclassified: {len(wrong)} total; {len(mis)} saved to misclassified_{tag}.csv (reasons are HEURISTIC).")
    display(mis.head(15))

    with open(OUTPUT_DIR / f"metrics_{tag}.json", "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=2, default=to_py)
    return m


ALL_RESULTS = {}
SKIP_NOTE = ("Model C intentionally skipped: no real conversation_id/message_order exists, so context_text == "
             "cleaned_text for every row and Model C would duplicate Model B (same input, same hyperparameters).")

## 5A. Baseline: TF-IDF + Logistic Regression

Input: `cleaned_text`. TF-IDF (word, 1–2-grams, 20,000 features) → Logistic Regression (`class_weight="balanced"`, `max_iter=1000`, `random_state=42`).
The token pattern is `\S+`, so that Telugu words, which contain combining vowel signs, and emojis stay as whole tokens. sklearn's default `\b\w\w+\b` splits Telugu words at vowel signs and drops emojis entirely.

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), max_features=20000,
                              token_pattern=r"(?u)\S+", lowercase=True)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
])
baseline.fit(train_df["cleaned_text"], train_df["label"])
val_pred = baseline.predict(val_df["cleaned_text"])
baseline_val_f1 = float(f1_score(val_df["label"], val_pred, average="macro"))
print(f"Baseline validation macro F1: {baseline_val_f1:.4f}")
joblib.dump(baseline, OUTPUT_DIR / "baseline_tfidf_logreg.joblib")

probs_baseline = baseline.predict_proba(test_df["cleaned_text"])
probs_baseline = probs_baseline[:, np.argsort(baseline.classes_)]   # align columns to ids 0,1,2
ALL_RESULTS["baseline"] = evaluate_model("baseline", "TF-IDF + Logistic Regression", "cleaned_text",
                                         test_df, probs_baseline, baseline_val_f1)

## 5B/5C. XLM-RoBERTa fine-tuning utilities

Both Transformer models use `XLMRobertaTokenizerFast`, `AutoModelForSequenceClassification(num_labels=3)`, `Trainer` and `TrainingArguments`, with the same hyperparameters (only the input column differs), so the comparison is fair:
5 epochs max, lr 2e-5, weight decay 0.01, warmup ratio 0.1, per-epoch evaluation/saving, best model by validation macro F1, early stopping patience 2, fp16 only on CUDA, seed 42.
Padding is dynamic per batch, with truncation at 128 tokens. This gives the same result as padding every input to 128 but runs much faster.

*Note: `transformers` v5 renamed `evaluation_strategy` to `eval_strategy` and merged `warmup_ratio` into `warmup_steps` (a float below 1 is read as a ratio). The code picks whichever names the installed version accepts.*

In [ ]:
def _load_metric(name):
    try:
        return evaluate.load(name)
    except Exception as e:  # offline fallback keeps the notebook runnable
        print(f"evaluate.load('{name}') failed ({e}); using scikit-learn equivalent.")
        return None

_METRICS = {n: _load_metric(n) for n in ["accuracy", "f1", "precision", "recall"]}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    def run(name, sk, **kw):
        m = _METRICS[name]
        if m is not None:
            return float(m.compute(predictions=preds, references=labels, **kw)[name])
        return float(sk(labels, preds, **kw))
    return {
        "accuracy": run("accuracy", accuracy_score),
        "macro_f1": run("f1", f1_score, average="macro"),
        "weighted_f1": run("f1", f1_score, average="weighted"),
        "precision": run("precision", precision_score, average="macro", zero_division=0),
        "recall": run("recall", recall_score, average="macro", zero_division=0),
    }


def make_training_args(output_dir):
    params = inspect.signature(TrainingArguments.__init__).parameters
    kw = dict(output_dir=str(output_dir), num_train_epochs=5, learning_rate=2e-5, weight_decay=0.01,
              save_strategy="epoch", load_best_model_at_end=True,
              metric_for_best_model="macro_f1", greater_is_better=True,
              per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
              fp16=USE_CUDA, seed=SEED, logging_strategy="epoch", save_total_limit=2,
              report_to="none", dataloader_num_workers=0)
    kw["evaluation_strategy" if "evaluation_strategy" in params else "eval_strategy"] = "epoch"
    # warmup_ratio=0.1: transformers v5 merged it into warmup_steps (a float < 1 is interpreted as a ratio)
    if "warmup_ratio" in params: kw["warmup_ratio"] = 0.1
    else: kw["warmup_steps"] = 0.1
    if "data_seed" in params: kw["data_seed"] = SEED
    return TrainingArguments(**kw)


def train_xlmr(tag, input_col, model_dir, tok_dir):
    seed_everything()
    tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)
    def to_ds(df):
        ds = datasets.Dataset.from_pandas(df[[input_col, "label"]].astype({input_col: object, "label": int}).rename(columns={input_col: "text", "label": "labels"}),
                                          preserve_index=False)
        return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH),
                      batched=True, remove_columns=["text"])
    tr, va, te = to_ds(train_df), to_ds(val_df), to_ds(test_df)
    n_trunc = int(sum(len(x) >= MAX_LENGTH for x in tr["input_ids"]))
    print(f"[{tag}] train/val/test = {len(tr)}/{len(va)}/{len(te)}; train inputs truncated at {MAX_LENGTH}: {n_trunc}")

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
    ckpt = CHECKPOINT_DIR / tag
    trainer = Trainer(model=model, args=make_training_args(ckpt), train_dataset=tr, eval_dataset=va,
                      processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                      compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()

    hist = pd.DataFrame([h for h in trainer.state.log_history if "eval_macro_f1" in h])
    print("Validation history per epoch:")
    display(hist[[c for c in ["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1", "eval_weighted_f1", "eval_precision", "eval_recall"] if c in hist]])
    best_val = float(trainer.state.best_metric)
    print(f"Best validation macro F1: {best_val:.4f}  (checkpoint: {trainer.state.best_model_checkpoint})")

    trainer.save_model(str(model_dir))            # best model (load_best_model_at_end=True)
    tokenizer.save_pretrained(str(tok_dir))
    hist.to_csv(Path(model_dir) / "validation_history.csv", index=False)

    logits = trainer.predict(te).predictions
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    if CLEANUP_CHECKPOINTS:
        shutil.rmtree(ckpt, ignore_errors=True)
    del trainer, model
    if USE_CUDA: torch.cuda.empty_cache()
    return probs, best_val

## 5B. XLM-RoBERTa **without** context

Input: `cleaned_text` (the current message only).

In [ ]:
probs_nc, best_val_nc = train_xlmr("no_context", "cleaned_text",
                                   OUTPUT_DIR / "xlm_roberta_no_context", OUTPUT_DIR / "tokenizer_no_context")
ALL_RESULTS["no_context"] = evaluate_model("no_context", "XLM-RoBERTa (no context)", "cleaned_text",
                                           test_df, probs_nc, best_val_nc)

## 5C. Context-aware XLM-RoBERTa (final model)

Input: `context_text`. This model is trained **only if** at least one message has earlier-message context. If there is no conversation metadata, `context_text` is identical to `cleaned_text` and Model C would duplicate Model B, so it is skipped. The skip is recorded in `metrics_context_aware.json` and in the comparison table.

In [ ]:
if RUN_CONTEXT_MODEL:
    probs_ca, best_val_ca = train_xlmr("context_aware", "context_text",
                                       OUTPUT_DIR / "xlm_roberta_context_aware", OUTPUT_DIR / "tokenizer_context_aware")
    ALL_RESULTS["context_aware"] = evaluate_model("context_aware", "XLM-RoBERTa (context-aware)", "context_text",
                                                  test_df, probs_ca, best_val_ca)
else:
    print("True context-aware training cannot be performed because conversation_id/message_order is unavailable.")
    print(SKIP_NOTE)
    with open(OUTPUT_DIR / "metrics_context_aware.json", "w", encoding="utf-8") as f:
        json.dump({"model": "XLM-RoBERTa (context-aware)", "status": "Skipped — no real conversation_id/message_order available.", "reason": SKIP_NOTE}, f, indent=2, ensure_ascii=False)

## 7. Model comparison and per-language results

In [ ]:
excluded_note = "; ".join(f"{k} excluded (see audit_report.json)" for k in excluded) or "all datasets used"
notes = {
    "baseline": f"Linear model on TF-IDF 1-2 grams, class_weight=balanced. {excluded_note}.",
    "no_context": f"Current message only. {excluded_note}.",
    "context_aware": f"Previous 2 messages + current ({n_ctx_rows} messages had earlier context). {excluded_note}.",
}

def fmt(v):
    return round(v, 4) if isinstance(v, float) else "N/A"

rows, lang_rows = [], []
for tag, m in ALL_RESULTS.items():
    pl = m["per_language"]
    rows.append({
        "Model name": m["model"], "Input used": m["input"],
        "Test accuracy": fmt(m["accuracy"]), "Test macro F1": fmt(m["macro_f1"]), "Test weighted F1": fmt(m["weighted_f1"]),
        "Telugu macro F1": fmt(pl["telugu"]["macro_f1"]), "English macro F1": fmt(pl["english"]["macro_f1"]),
        "Code-mixed macro F1": fmt(pl["code_mixed"]["macro_f1"]),
        "Best validation macro F1": fmt(m["best_validation_macro_f1"]), "Notes": notes[tag],
    })
    for lang, v in pl.items():
        lang_rows.append({"model": m["model"], "language_type": lang, "n_test": v["n"], "status": v["status"],
                          "accuracy": fmt(v["accuracy"]), "macro_f1": fmt(v["macro_f1"]), "weighted_f1": fmt(v["weighted_f1"])})

if "context_aware" not in ALL_RESULTS:
    rows.append({"Model name": "XLM-RoBERTa (context-aware)", "Input used": "context_text",
                 **{k: "Skipped — no real conversation_id/message_order available." for k in ["Test accuracy", "Test macro F1", "Test weighted F1", "Telugu macro F1",
                                           "English macro F1", "Code-mixed macro F1", "Best validation macro F1"]},
                 "Notes": SKIP_NOTE})
comparison = pd.DataFrame(rows)
per_language = pd.DataFrame(lang_rows)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")
per_language.to_csv(OUTPUT_DIR / "per_language_results.csv", index=False, encoding="utf-8-sig")
display(comparison)
display(per_language)

audit_report["results_summary"] = comparison.to_dict(orient="records")
with open(OUTPUT_DIR / "audit_report.json", "w", encoding="utf-8") as f:
    json.dump(audit_report, f, ensure_ascii=False, indent=2, default=to_py)

required = ["audit_report.json", "cleaned_all_data.csv", "train.csv", "validation.csv", "test.csv", "label_mapping.json",
            "baseline_tfidf_logreg.joblib", "xlm_roberta_no_context", "xlm_roberta_context_aware",
            "tokenizer_no_context", "tokenizer_context_aware", "metrics_baseline.json", "metrics_no_context.json",
            "metrics_context_aware.json", "confusion_matrix_baseline.png", "confusion_matrix_no_context.png",
            "confusion_matrix_context_aware.png", "misclassified_baseline.csv", "misclassified_no_context.csv",
            "misclassified_context_aware.csv", "model_comparison.csv", "per_language_results.csv"]
if not RUN_CONTEXT_MODEL:   # intentionally not produced when Model C is skipped
    required = [r for r in required if r not in {"xlm_roberta_context_aware", "tokenizer_context_aware",
                                                 "confusion_matrix_context_aware.png", "misclassified_context_aware.csv"}]
missing = [r for r in required if not (OUTPUT_DIR / r).exists()]
print("All required outputs present." if not missing else f"MISSING OUTPUTS: {missing}")

## 8. Explanation

### The three models
- **TF-IDF + Logistic Regression.** A fast baseline. It turns words and word pairs into numerical importance scores (frequent in this message, rare across the corpus → high score), then predicts sentiment with a linear classifier. It has no notion of word meaning beyond the exact tokens it saw in training.
- **XLM-RoBERTa without context.** A pretrained multilingual Transformer (trained on 100 languages, including Telugu). Its subword tokenizer and contextual representations handle Telugu script, English, Romanized words and mixed-language text better than TF-IDF, but it **sees only the current message**.
- **Context-aware XLM-RoBERTa.** Reads the previous two messages plus the current one (`[PREVIOUS_2] … [PREVIOUS_1] … [CURRENT] …`). This helps it interpret short, context-dependent replies such as "ledu", "okay", "fine", "avvale" or "not really", whose sentiment depends on what was said before. *It is trained only when the data has real conversation metadata. Without it, Model C would be identical to Model B, so it is skipped (see Section 3 and the comparison table).*

### Training parameters
| Parameter | Meaning |
|---|---|
| `max_length=128` | Maximum number of tokens per message/context; longer inputs are truncated. |
| `epochs=5` | Maximum number of full passes over the training data. |
| `learning_rate=2e-5` | A small update step suited to fine-tuning a pretrained Transformer without destroying what it already knows. |
| `weight_decay=0.01` | Penalises large weights to reduce overfitting. |
| `warmup_ratio=0.1` | The learning rate ramps up gradually over the first 10% of steps, which stabilises early training. |
| `early_stopping_patience=2` | Training stops if validation macro F1 does not improve for two consecutive evaluations. |
| `class_weight="balanced"` | Makes the baseline weight errors on rarer classes more heavily, to handle unequal class sizes. |
| **Macro F1** | Average of the per-class F1 scores, so Negative, Neutral and Positive count equally regardless of size. |
| **Weighted F1** | Per-class F1 weighted by class frequency, so it reflects the actual class distribution. |
| **Accuracy** | Percentage of all test predictions that are correct. |

### Error analysis caveat
`likely_error_reason` in the `misclassified_*.csv` files is a **rule-based heuristic**: keyword, emoji, script-mixing, confidence and annotator-disagreement rules. It is meant to guide manual inspection. It is **not** a verified cause of the error.

In [ ]:
# Final summary, printed only from results computed above in this run
print("DATA USED:", eligible, "| EXCLUDED:", list(excluded))
for k, v in excluded.items():
    print(f"  - {k}: {v}")
print("True conversational context available:", TRUE_CONTEXT, "| Model C trained:", RUN_CONTEXT_MODEL)
print()
print(comparison[["Model name", "Input used", "Test accuracy", "Test macro F1", "Test weighted F1",
                  "Telugu macro F1", "English macro F1", "Code-mixed macro F1", "Best validation macro F1"]].to_string(index=False))

## 9. Final report (written automatically)

In [ ]:
lines = []
lines.append("# Multilingual sentiment: Models A and B (Colab GPU run)\n")
lines.append(f"Run folder: `{OUTPUT_DIR}`  \nGPU: {torch.cuda.get_device_name(0)}\n")
lines.append(f"Inputs: all {len(EXPECTED_SHA256)} files matched the audited SHA-256 checksums.\n")
lines.append("## Data after cleaning\n")
lines.append(pd.DataFrame(cleaning_stats).T.to_markdown() + "\n")
lines.append(f"Split ({split_method}): train {len(train_df)}, validation {len(val_df)}, test {len(test_df)}\n")
lines.append("## Model comparison (test set)\n")
lines.append(comparison.to_markdown(index=False) + "\n")
lines.append("## Per-language results (test set)\n")
lines.append(per_language.to_markdown(index=False) + "\n")
for tag, m in ALL_RESULTS.items():
    lines.append(f"## {m['model']}: per-class (test set)\n")
    lines.append(pd.DataFrame(m["per_class"]).T.round(4).to_markdown() + "\n")
    cm_df = pd.read_csv(OUTPUT_DIR / f"confusion_matrix_{tag}.csv", index_col=0)
    lines.append("Confusion matrix (rows = actual):\n\n" + cm_df.to_markdown() + "\n")
    lines.append(f"![confusion matrix](confusion_matrix_{tag}.png)\n")
lines.append("## Model C\n")
lines.append("Skipped — no real conversation_id/message_order available.\n")
lines.append("\nError reasons in misclassified_*.csv are rule-based heuristics, not ground truth.\n")
(OUTPUT_DIR / "final_report.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))
print("\nRUN COMPLETE. Everything is saved in:", OUTPUT_DIR)